<a href="https://colab.research.google.com/github/mircomarahrens/nconpp/blob/py-dev/python/notebooks/00_tensornetworks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tensor network library

In [10]:
from nconpp import Graph

g = Graph(5, False)
print(g.num_vertices)
print(g.get_vertices())
print(g.vertices[0].edge_indices)
print(g.remove_vertex(0))
print(g.get_vertices())
print(g.add_vertex(5))
print(g.get_vertices())
print(g.add_edge(0,1,2))
print(g.get_edges())
print(g.num_edges)
print(g.edges[0].src, g.edges[0].dest)
print(g.remove_edge(0))
print(g.num_edges)
print(g.get_edges())

5
{0, 1, 2, 3, 4}
set()
0
{1, 2, 3, 4}
5
{1, 2, 3, 4, 5}
0
{0}
1
1 2
0
0
set()


In [11]:
# g.add_edge(1,1,2)) # throws ValueError: Parallel edge already present.

In [12]:
import numpy as np
from nconpp import TensorNetwork

tensorList = [
    (np.random.randn(60)+1j*np.random.randn(60)).reshape(3,4,5),
    (np.random.randn(3780)+1j*np.random.randn(3780)).reshape(5, 3, 6, 7, 6),
    (np.random.randn(14)+1j*np.random.randn(14)).reshape(7,2),
    (np.random.randn(8)+1j*np.random.randn(8)).reshape(8),
    (np.random.randn(72)+1j*np.random.randn(72)).reshape(8,9),
    (np.random.randn(81)+1j*np.random.randn(81)).reshape(9,9)
]

legsList = [
    [3, -2, 2],
    [2, 3, 1, 4, 1],
    [4, -1],
    [5],
    [5, -3],
    [6, 6]
]

tn = TensorNetwork(tensorList, legsList)

print(tn.num_tensors)
tn.contract()
print(tn.num_tensors)
tn.connect()
print(tn.num_tensors)

6
3
1


In [15]:
%matplotlib widget
from ipywidgets import interact
import matplotlib.pyplot as plt
import numpy as np

from nconpp import LatticeGraph

# lg = LatticeGraph("honeycomb", [4,4], [((0, -1), (0, +1), (+1, 0)), ((-1, 0), (0, -1), (0, +1))], ("pbc", "pbc"))


# plot
@interact(lx=4, ly=4, boundary=True)
def plot_lattice(lx, ly, boundary):
    lg = LatticeGraph("honeycomb", [lx, ly])

    fig, ax = plt.figure(), plt.axes()

    # vertices
    for i in lg.vertices:
        v = lg.vertices[i]
        if v.boundary:
            ax.scatter(v.coordinate[0], v.coordinate[1], c="r", alpha=0.5)
        else:
            ax.scatter(v.coordinate[0], v.coordinate[1], c="b", alpha=0.5)
        ax.annotate(i, v.coordinate)

    # edges
    for i in lg.edges:
        e = lg.edges[i]
        e_src = lg.vertices[e.src].coordinate
        e_dest = lg.vertices[e.dest].coordinate
        x, y = np.array([e_src, e_dest]).T
        if e.boundary and boundary:
            ax.plot(x, y, c="forestgreen", linestyle=":", alpha=0.8)
            ax.annotate(
                i,
                xy=(np.mean(x) - 0.1, np.mean(y) - 0.1),
                ha="center",
                va="center",
                color="forestgreen",
                fontsize=6,
            )
        else:
            ax.plot(x, y, c="lightsteelblue", linestyle="-", alpha=0.8)
            ax.annotate(
                i,
                xy=(np.mean(x) + 0.1, np.mean(y) + 0.1),
                ha="center",
                va="center",
                color="lightsteelblue",
                fontsize=6,
            )
    plt.show()

    print(f'e_id\tsrc\tdest\tboundary')
    for i in lg.edges:
        e = lg.edges[i]
        e_src = lg.vertices[e.src].coordinate
        e_dest = lg.vertices[e.dest].coordinate
        print(f'{i}\t{e_src}\t{e_dest}\t{e.boundary}')

interactive(children=(IntSlider(value=4, description='lx', max=12, min=-4), IntSlider(value=4, description='ly…

In [16]:
from matplotlib import pyplot as plt
from functools import partial
from ipywidgets import interact

import numpy as np

def f(x): return 3*x**2 + 2*x + 1

def plot_function(f, title=None, min=-2.1, max=2.1, color='r', ylim=None):
    x = np.linspace(min,max, 100)[:,None]
    if ylim: plt.ylim(ylim)
    plt.plot(x, f(x), color)
    if title is not None: plt.title(title)

def quad(a,b,c,x):
    return a*x**2 + b*x + c

def mk_quad(a,b,c): return partial(quad, a,b,c)
def noise(x, scale): return np.random.normal(scale=scale, size=x.shape)
def add_noise(x, mult, add): return x * (1+noise(x,mult)) + noise(x,add)

np.random.seed(42)

x = np.linspace(-2, 2)[:,None]
y = add_noise(f(x), 0.15, 1.5)

@interact(a=1.5, b = 1.5, c =1.5)
def plot_quad(a,b,c):
    plt.scatter(x,y)
    plot_function(mk_quad(a,b,c), ylim=(-3, 12))

interactive(children=(FloatSlider(value=1.5, description='a', max=4.5, min=-1.5), FloatSlider(value=1.5, descr…

# MPO